# Performance Leaderboard

This page provides an interactive view of the latest experimental results across different datasets and models. 

Results are aggregated from multiple runs and compared against the **OPLS-DA** baseline using a paired t-test.

In [ ]:
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from pathlib import Path
import os

# Load data generated by docs/generate_leaderboard_docs.py
csv_path = Path('_static') / 'leaderboard_data.csv'
if not csv_path.exists():
    # Fallback for local execution if running inside docs/source
    csv_path = Path('.') / '_static' / 'leaderboard_data.csv'

if csv_path.exists():
    df = pd.read_csv(csv_path)
else:
    # Placeholder data if the CSV is missing (e.g. during a build without results)
    print('Warning: No leaderboard data found.')
    df = pd.DataFrame(columns=['Dataset', 'Method', 'Train', 'Test'])

# Consistent coloring function
def get_color_map(methods):
    colors = px.colors.qualitative.Plotly + px.colors.qualitative.Bold
    unique_methods = sorted(list(set(methods)))
    return {m: colors[i % len(colors)] for i, m in enumerate(unique_methods)}

if not df.empty:
    color_map = get_color_map(df['Method'].unique())
else:
    color_map = {}

## Summary Table

| Symbol | Meaning |
|---|---|
| `+` | Significantly better than OPLS-DA |
| `-` | Significantly worse than OPLS-DA |
| `≈` | No significant difference |

In [ ]:
if not df.empty:
    preferred_cols = ['Dataset', 'Method', 'Train', 'Test', 'Sig Te', 'Baseline']
    actual_cols = [c for c in preferred_cols if c in df.columns]
    styled_df = df[actual_cols].sort_values(['Dataset', 'Test'], ascending=[True, False])
    from IPython.display import display, HTML
    display(HTML(styled_df.to_html(index=False)))
else:
    print('No data available to display.')

## Detailed Results by Dataset

In [ ]:
if not df.empty:
    for ds in df['Dataset'].unique():
        ds_df = df[df['Dataset'] == ds].sort_values('Test', ascending=False)
        fig = px.bar(
            ds_df, x='Method', y='Test', error_y='Test Std' if 'Test Std' in ds_df.columns else None,
            title=f"Leaderboard: {ds.upper()}", 
            color='Method', 
            color_discrete_map=color_map,
            template='plotly_white',
            labels={'Test': 'Balanced Accuracy'}
        )
        fig.update_layout(yaxis_range=[0, 1.05])
        fig.show()
else:
    print('No results available.')